# Docent DQL Notebook
Fetches agent runs via DQL and loads them into pandas.

### README
- Requires `pip install docent-python pandas`
- API key is prefilled; rotate in Docent if you regenerate keys.
- Run the cells as-is to fetch data into pandas.


In [ ]:
%pip install -q docent-python pandas


In [ ]:
from docent import Docent
import pandas as pd

API_KEY = "dk_OiQtvxsaRPoPQ3xw_aggcGTjoUiLIZsWLgrJoUBK0B4vY18MCiXSsSjcwdbmE64"
SERVER_URL = "http://localhost:8890"
COLLECTION_ID = "7d99d911-8bfd-44f1-bb06-ca4ad59c5c3e"

DQL_QUERY = """
SELECT
  ar.id AS agent_run_id,
  ar.metadata_json->'agent'->>'name' AS "metadata.agent",
  ar.metadata_json->'agent'->>'model_name' AS "metadata.agent.model_name",
  ar.metadata_json->'scores'->>'reward' AS "metadata.scores.reward",
  ar.metadata_json->>'task' AS "metadata.task",
  ar.created_at AS created_at
FROM agent_runs ar
ORDER BY ar.created_at ASC
"""

client = Docent(api_key=API_KEY, server_url=SERVER_URL)
result = client.execute_dql(COLLECTION_ID, DQL_QUERY)
df = client.dql_result_to_df_experimental(result)

df


In [ ]:
# Drop rows with empty reward or task
df = df.dropna(subset=['metadata.scores.reward', 'metadata.task'])
df


In [ ]:
import matplotlib.pyplot as plt

# Convert reward to numeric
df['metadata.scores.reward'] = pd.to_numeric(df['metadata.scores.reward'])

# Calculate mean reward for each model
reward_by_model = df.groupby('metadata.agent.model_name')['metadata.scores.reward'].mean()

# Plot bar chart
plt.figure(figsize=(10, 6))
reward_by_model.plot(kind='bar')
plt.xlabel('Model Name')
plt.ylabel('Mean Reward')
plt.title('Mean Reward by Model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Create pivot table with mean and variance of reward for each task and model
pivot_mean = df.pivot_table(
    values='metadata.scores.reward',
    index='metadata.task',
    columns='metadata.agent.model_name',
    aggfunc='mean'
)

print("Mean Reward by Task and Model:")
display(pivot_mean)

In [ ]:
# Scatter plot comparing gpt-5-codex vs gpt-5.1-codex rewards by task
# Group by (x, y) pairs and count occurrences for point sizing
scatter_data = pivot_mean[['openai/gpt-5-codex', 'openai/gpt-5.1-codex']].dropna()
grouped = scatter_data.groupby([scatter_data['openai/gpt-5-codex'], scatter_data['openai/gpt-5.1-codex']]).size().reset_index(name='count')

plt.figure(figsize=(10, 10))
plt.scatter(grouped['openai/gpt-5-codex'], grouped['openai/gpt-5.1-codex'], 
            s=grouped['count'] * 50, alpha=0.7)

# Add diagonal line for reference
plt.plot([0, 1], [0, 1], 'r--', label='Equal performance')

# Add annotations for point sizes
for _, row in grouped.iterrows():
    plt.annotate(f"n={int(row['count'])}", 
                 (row['openai/gpt-5-codex'], row['openai/gpt-5.1-codex']),
                 textcoords="offset points", xytext=(5, 5), fontsize=8)

plt.xlabel('openai/gpt-5-codex Mean Reward')
plt.ylabel('openai/gpt-5.1-codex Mean Reward')
plt.title('Task Performance: gpt-5-codex vs gpt-5.1-codex')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Convert rewards to categories (thirds: low, medium, high)
def categorize_reward(value):
    if value < 1/3:
        return 'Low (0-0.33)'
    elif value < 2/3:
        return 'Medium (0.33-0.67)'
    else:
        return 'High (0.67-1.0)'

# Apply categorization to both models
scatter_data = pivot_mean[['openai/gpt-5-codex', 'openai/gpt-5.1-codex']].dropna()
gpt5_categories = scatter_data['openai/gpt-5-codex'].apply(categorize_reward)
gpt51_categories = scatter_data['openai/gpt-5.1-codex'].apply(categorize_reward)

# Create confusion matrix
from sklearn.metrics import confusion_matrix

categories = ['Low (0-0.33)', 'Medium (0.33-0.67)', 'High (0.67-1.0)']
cm = confusion_matrix(gpt5_categories, gpt51_categories, labels=categories)

# Plot confusion matrix using matplotlib
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax)

# Set tick labels
ax.set(xticks=range(len(categories)),
       yticks=range(len(categories)),
       xticklabels=categories,
       yticklabels=categories,
       xlabel='gpt-5.1-codex Performance',
       ylabel='gpt-5-codex Performance',
       title='Confusion Matrix: Task Performance Comparison\n(gpt-5-codex vs gpt-5.1-codex)')

# Rotate x tick labels for better readability
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add text annotations
thresh = cm.max() / 2.
for i in range(len(categories)):
    for j in range(len(categories)):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nTotal tasks compared: {len(scatter_data)}")
print(f"\nTasks where gpt-5.1-codex improved: {sum(gpt51_categories.values > gpt5_categories.values)}")
print(f"Tasks where gpt-5.1-codex regressed: {sum(gpt51_categories.values < gpt5_categories.values)}")
print(f"Tasks with same category: {sum(gpt51_categories.values == gpt5_categories.values)}")


In [ ]:
# Filter to tasks where gpt-5-codex outperforms gpt-5.1-codex
gpt5_better = pivot_mean[pivot_mean['openai/gpt-5-codex'] > pivot_mean['openai/gpt-5.1-codex']]
print(f"Tasks where gpt-5-codex outperforms gpt-5.1-codex: {len(gpt5_better)}")
print(gpt5_better[['openai/gpt-5-codex', 'openai/gpt-5.1-codex']])
print(f"\nTask IDs: {gpt5_better.index.tolist()}")


In [ ]:
task_ids = gpt5_better.index.tolist()

In [ ]:
df_subset = df[(df['metadata.task'].isin(task_ids)) & (df['metadata.agent.model_name'] == 'openai/gpt-5.1-codex')]
df_subset

In [ ]:
agent_run_ids = df_subset['agent_run_id'].tolist()

In [ ]:
for agent_run_id in agent_run_ids:
    client.tag_transcript(COLLECTION_ID, agent_run_id, "gpt-5-codex-worse")